In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

def local_reaching_centrality_true(G, v):
    """
    True Mones, Vicsek & Vicsek (2012) local reaching centrality:
    C(v) = (1 / (N-1)) * sum_{u reachable from v, u != v} 1 / d(v, u)
    where d(v,u) is the shortest-path (hop) distance from v to u.
    Unlike nx.local_reaching_centrality with weight=None, this does NOT
    collapse to a plain reachability proportion -- it keeps the
    inverse-distance weighting for every reachable node.
    """
    N = len(G)
    if N <= 1:
        return 0.0
    lengths = nx.single_source_shortest_path_length(G, v)  # {node: hop distance}
    total = sum(1.0 / d for u, d in lengths.items() if u != v and d > 0)
    return total / (N - 1)

def global_reaching_centrality_true(G):
    if len(G) == 0:
        return 0.0
    lrc = {v: local_reaching_centrality_true(G, v) for v in G.nodes()}
    c_max = max(lrc.values())
    return sum(c_max - c for c in lrc.values()) / (len(G) - 1)
# Initialize lists to store results
average_shortest_paths = []
network_densities = []
average_clustering_coefficients = []
global_reaching_centralities = []

# Loop through the 12 datasets
for i in range(1, 13):
    # Load the dataset
    df = pd.read_csv(f'TMin{i}.csv', index_col=0)
    # Create a directed graph
    G = nx.DiGraph()
    G.add_nodes_from(df.index)
    
    # Add edges based on the DataFrame
    for source in df.index:
        for target in df.columns:
            if df.loc[source, target] == 1:
                G.add_edge(target, source)
    
    # Calculate network metrics
    try:
        avg_shortest_path = nx.average_shortest_path_length(G) if nx.is_strongly_connected(G) else float('inf')
    except nx.NetworkXError:
        avg_shortest_path = float('inf')  # Handle disconnected graphs
    
    network_density = nx.density(G)
    avg_clustering_coeff = nx.average_clustering(G.to_undirected())
    grc = global_reaching_centrality(G)
    
    # Append metrics to the respective lists
    average_shortest_paths.append(avg_shortest_path)
    network_densities.append(network_density)
    average_clustering_coefficients.append(avg_clustering_coeff)
    global_reaching_centralities.append(grc)
    
    # Generate the graph visualization
    pos = nx.spring_layout(G, k=2.95)
    plt.figure(figsize=(25, 20))
    nx.draw(
        G, 
        pos, 
        with_labels=True, 
        node_size=8000, 
        node_color='lightgreen', 
        edge_color='black', 
        arrows=True, 
        arrowsize=15, 
        font_color='darkblue', 
        font_size=12, 
        font_weight='bold'
    )
    
    # Save the graph with a unique title
    plt.title(f"Network Visualization TM{i}", fontsize=20, fontweight='bold', color='darkgreen')
    plt.axis('off')
    plt.savefig(f"Network_TM{i}.png", format='png', bbox_inches='tight')
    plt.close()

# Save metrics to a CSV file
results = pd.DataFrame({
    'Dataset': [f'TMin{i}' for i in range(1, 13)],
    'Average Shortest Path': average_shortest_paths,
    'Network Density': network_densities,
    'Average Clustering Coefficient': average_clustering_coefficients,
    'Global Reaching Centrality': global_reaching_centralities
})
results.to_csv('UPNetwork_Metrics_in.csv', index=False)

print("Networks saved and metrics calculated. Results saved to 'Network_Metrics.csv'.")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Load the metrics from the CSV file
results = pd.read_csv('UPNetwork_Metrics_in.csv')

# Define the time segments
datasets = [
    "Jan-Apr20", "May-Aug20", "Sep-Dec20", "Jan-Apr21", "May-Aug21", "Sep-Dec21", 
    "Jan-Apr22", "May-Aug22", "Sep-Dec22", "Jan-Apr23", "May-Aug23", "Sep-Dec23"
]

# Extract metrics
average_shortest_paths = results['Average Shortest Path']
network_densities = results['Network Density']
average_clustering_coefficients = results['Average Clustering Coefficient']
global_reaching_centralities = results['Global Reaching Centrality']

# Create the 2x2 subplot figure
fig, axs = plt.subplots(2, 2, figsize=(10,7))  # 2 rows, 2 columns

# Plot Average Shortest Path
axs[0, 0].plot(datasets, average_shortest_paths, marker='o', linestyle=':', color='blue')
axs[0, 0].set_title('(a) Average Shortest Path', fontsize=14, fontweight='bold')
axs[0, 0].set_xticklabels(datasets, rotation=45)
axs[0, 0].grid(alpha=0.5)

# Plot Network Density
axs[0, 1].plot(datasets, network_densities, marker='o', linestyle='-.', color='red')
axs[0, 1].set_title('(b) Network Density', fontsize=14, fontweight='bold')
axs[0, 1].set_xticklabels(datasets, rotation=45)
axs[0, 1].grid(alpha=0.5)

# Plot Average Clustering Coefficient
axs[1, 0].plot(datasets, average_clustering_coefficients, marker='s', linestyle='--', color='orange')
axs[1, 0].set_title('(c) Average Clustering Coefficient', fontsize=14, fontweight='bold')
axs[1, 0].set_xticklabels(datasets, rotation=45)
axs[1, 0].grid(alpha=0.5)

# Plot Global Reaching Centrality
axs[1, 1].plot(datasets, global_reaching_centralities, marker='^', linestyle='-', color='green')
axs[1, 1].set_title('(d) Global Reaching Centrality', fontsize=14, fontweight='bold')
axs[1, 1].set_xticklabels(datasets, rotation=45)
axs[1, 1].grid(alpha=0.5)

# Adjust layout and save
plt.tight_layout()
plt.savefig('UPNetwork_Metrics_CombinedRev.png')
plt.show()
